# Intro to ML and Linear Regression (Enhanced)

This notebook is based on the [Google ML Crash Course: Linear Regression](https://developers.google.com/machine-learning/crash-course/linear-regression). It walks through the fundamentals of Linear Regression — the simplest and most foundational supervised learning algorithm.

---

## 🚗 The Problem: Predicting Taxi Fares

We are trying to solve a real-world problem: **How much should a taxi ride cost?**

We have data on **Trip Miles** (the distance traveled) and we want to predict the **Fare** (the cost). This is a classic regression problem because the output (Fare) is a continuous numerical value.

### Why Trip Miles?
In a perfect world, a taxi fare is directly proportional to the distance. By finding the relationship between miles and fare, we can build a model that predicts the cost of any future trip given its distance.

---

## 📉 Loss Functions and Optimization

### What is Loss?
**Loss** is a numerical value that tells us how wrong the model's prediction is. If the prediction is perfect, loss is 0. Our goal is to minimize this loss.

### Types of Loss
1.  **Mean Squared Error (MSE)**: The average of the squared differences between the predicted and actual values. It penalizes large errors heavily.
    $$MSE = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$$
2.  **Mean Absolute Error (MAE)**: The average of the absolute differences. It is less sensitive to outliers than MSE.
    $$MAE = \frac{1}{N} \sum_{i=1}^{N} |y_i - \hat{y}_i|$$

### 🔄 Gradient Descent
**Gradient Descent** is the algorithm used to minimize the loss. It calculates the "gradient" (the slope) of the loss function with respect to the weights and moves the weights in the opposite direction (downhill) to find the minimum loss.

#### Simple Python Example of Gradient Descent Update:
```python
# w = current weight, lr = learning rate, gradient = derivative of loss
new_weight = w - (learning_rate * gradient)
```

### 🏁 Convergence
**Convergence** occurs when the loss function stops changing significantly between iterations. This indicates that the algorithm has found a local or global minimum and the model has finished "learning."

---

## ⚙️ Hyperparameters

Hyperparameters are the "knobs" we turn to control how the model trains. Unlike weights and bias, which the model learns itself, we must set these manually.

- 🎓 **Learning Rate**: Controls the size of the steps taken during gradient descent. 
    - Too high: The model might skip over the minimum (unstable).
    - Too low: The model will take forever to learn.
- 🔄 **Epochs**: The number of times the model scans the *entire* dataset.
- 📦 **Batch Size**: The number of training examples used in one single gradient update.

In [ ]:
# ──────────────────────────────────────────────────────
# Setup and Data Loading
# ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

# Load the Chicago Taxi dataset
DATA_URL = 'https://download.mlcc.google.com/mledu-datasets/chicago_taxi_train.csv'
chicago_taxi_dataset = pd.read_csv(DATA_URL)

# Select relevant columns for our experiments
training_df = chicago_taxi_dataset.loc[:, ('TRIP_MILES', 'TRIP_SECONDS', 'FARE')].dropna()

print(f"Dataset loaded: {training_df.shape[0]:,} rows")
training_df.head()

# Exploratory Data Analysis

Before building any model, it's important to **visualize the data**.

By plotting `TRIP_MILES` against `FARE`, we can immediately see if a linear relationship exists and identify any outliers.

In [ ]:
# Visualize correlation
plt.figure(figsize=(10, 6))
plt.scatter(training_df['TRIP_MILES'], training_df['FARE'], alpha=0.1, s=10, color='steelblue')
plt.xlabel('Trip Miles', fontsize=13)
plt.ylabel('Fare ($)', fontsize=13)
plt.title('Chicago Taxi: Trip Miles vs Fare Correlation', fontsize=15)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Building and Training the Model

We use **`tf.keras.Sequential`** to build our linear regression model.

The model uses a `Dense` layer with a single neuron to implement the line equation:

$$\text{FARE} = w \times \text{FEATURE} + b$$

- **Optimizer**: `RMSprop` — an adaptive learning rate optimizer.
- **Loss**: `Mean Squared Error (MSE)`.
- **Metrics**: `Root Mean Squared Error (RMSE)` — provides error in the same units as the label ($).

In [ ]:
def build_model(learning_rate, feature_count):
    """Create and compile a simple linear regression model."""
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=1, input_shape=(feature_count,))
    ])

    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=learning_rate),
        loss='mean_squared_error',
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )
    return model

def train_model(model, df, feature_names, label_name, epochs, batch_size):
    """Train the model on the provided features and label."""
    history = model.fit(
        x=df[feature_names],
        y=df[label_name],
        batch_size=batch_size,
        epochs=epochs,
        verbose=0
    )
    return history

def plot_loss_curve(history, label):
    """Plot the loss (RMSE) over epochs."""
    plt.plot(history.history['root_mean_squared_error'], label=label)
    plt.xlabel('Epoch')
    plt.ylabel('RMSE')
    plt.legend()

## 🧪 Experiment 1: Baseline Model (One Feature)

We start by training a model using only `TRIP_MILES` to see how well it predicts the fare.

In [ ]:
learning_rate = 0.001
epochs = 20
batch_size = 50
feature_names = ['TRIP_MILES']
label_name = 'FARE'

model_1 = build_model(learning_rate, len(feature_names))
history_1 = train_model(model_1, training_df, feature_names, label_name, epochs, batch_size)

plt.figure(figsize=(8, 5))
plot_loss_curve(history_1, 'Baseline (Miles)')
plt.title('Experiment 1: Baseline Loss Curve')
plt.show()

## 🧪 Experiment 2: Hyperparameter Tuning

Hyperparameters significantly affect training. Here we compare a low learning rate (stable) with a much higher one (potentially unstable).

In [ ]:
learning_rate_high = 0.1 

model_2 = build_model(learning_rate_high, len(feature_names))
history_2 = train_model(model_2, training_df, feature_names, label_name, epochs, batch_size)

plt.figure(figsize=(8, 5))
plot_loss_curve(history_1, 'LR = 0.001 (Stable)')
plot_loss_curve(history_2, 'LR = 0.1 (Faster but Unstable)')
plt.title('Experiment 2: Effect of Learning Rate')
plt.show()

## 🧪 Experiment 3: Feature Engineering (Adding Minutes)

Sometimes raw data isn't in the best format. We'll convert `TRIP_SECONDS` to `TRIP_MINUTES` to see if adding time information improves our predictions.

In [ ]:
# Feature Engineering
training_df['TRIP_MINUTES'] = training_df['TRIP_SECONDS'] / 60

multi_features = ['TRIP_MILES', 'TRIP_MINUTES']

model_3 = build_model(0.001, len(multi_features))
history_3 = train_model(model_3, training_df, multi_features, label_name, epochs, batch_size)

plt.figure(figsize=(8, 5))
plot_loss_curve(history_1, 'Miles Only')
plot_loss_curve(history_3, 'Miles + Minutes')
plt.title('Experiment 3: Adding More Features')
plt.show()

## ✅ Model Accuracy: Predicted vs Observed

To evaluate how well our model performs on real data, we plot the **Observed Fare** against our **Predicted Fare**. A perfect model would align all points on the diagonal red line.

In [ ]:
# Make predictions
sample_df = training_df.sample(500)
predictions = model_3.predict(sample_df[multi_features])

plt.figure(figsize=(10, 6))
plt.scatter(sample_df['FARE'], predictions, alpha=0.5, color='seagreen')
plt.plot([0, 80], [0, 80], 'r--', label='Perfect Prediction')
plt.xlabel('Observed Fare ($)', fontsize=13)
plt.ylabel('Predicted Fare ($)', fontsize=13)
plt.title('Accuracy Check: Observed vs. Predicted', fontsize=15)
plt.legend()
plt.grid(True)
plt.show()

final_rmse = history_3.history['root_mean_squared_error'][-1]
print(f"Final Model RMSE: {final_rmse:.2f} (Average error is about ${final_rmse:.2f})")